# dbAppsUA08: Створення та керування даними
## Покроковий довідник

У цьому уроці ви навчитесь **створювати базу даних з нуля**: створювати таблиці, вставляти дані, оновлювати записи, видаляти рядки та застосовувати обмеження.

## Налаштування

In [ ]:
import pandas as pd
import sqlite3

# Створюємо підключення до НОВОЇ бази даних
conn = sqlite3.connect("practice.db")

print("Підключення встановлено.")
print("Працюємо з: practice.db")

## Розділ 1: CREATE TABLE та INSERT

In [ ]:
# Створюємо таблицю 'students' (студенти)
createStudentsTable = """
CREATE TABLE students (
    studentId INTEGER PRIMARY KEY,
    firstName TEXT,
    lastName TEXT,
    grade INTEGER,
    gpa REAL
);
"""

conn.execute(createStudentsTable)
conn.commit()

print("Таблицю 'students' створено успішно.")

In [ ]:
# Вставляємо записи студентів
conn.execute("INSERT INTO students (firstName, lastName, grade, gpa) VALUES ('Alice', 'Johnson', 10, 3.85)")
conn.commit()

conn.execute("INSERT INTO students (firstName, lastName, grade, gpa) VALUES ('Bob', 'Smith', 11, 3.45)")
conn.commit()

conn.execute("INSERT INTO students (firstName, lastName, grade, gpa) VALUES ('Charlie', 'Brown', 9, 2.90)")
conn.commit()

conn.execute("INSERT INTO students (firstName, lastName, grade, gpa) VALUES ('Diana', 'Prince', 12, 3.95)")
conn.commit()

print("4 записи студентів вставлено.")

In [ ]:
# Перевіряємо дані з SELECT
studentData = pd.read_sql("SELECT * FROM students", conn)
print("Всі студенти:")
print(studentData)

## Розділ 2: UPDATE та DELETE

**Золоте правило:** Завжди тестуйте WHERE з SELECT **перед** UPDATE або DELETE!

Це називається **test-first підхід** (спочатку тестуємо).

In [ ]:
# КРОК 1: Тестуємо WHERE з SELECT
testResult = pd.read_sql("SELECT * FROM students WHERE firstName = 'Alice'", conn)
print("Записи для оновлення:")
print(testResult)

In [ ]:
# КРОК 2: Виконуємо UPDATE
conn.execute("UPDATE students SET gpa = 3.92 WHERE firstName = 'Alice'")
conn.commit()
print("Оновлення виконано.")

# КРОК 3: Перевіряємо
result = pd.read_sql("SELECT * FROM students WHERE firstName = 'Alice'", conn)
print("Оновлений запис Alice:")
print(result)

In [ ]:
# DELETE: теж спочатку тестуємо!
# КРОК 1: Тестуємо
testDelete = pd.read_sql("SELECT * FROM students WHERE firstName = 'Charlie'", conn)
print("Записи для видалення:")
print(testDelete)

In [ ]:
# КРОК 2: Видаляємо
conn.execute("DELETE FROM students WHERE firstName = 'Charlie'")
conn.commit()
print("Видалення виконано.")

# КРОК 3: Перевіряємо
result = pd.read_sql("SELECT * FROM students", conn)
print("Всі студенти (Charlie видалено):")
print(result)

### НЕБЕЗПЕКА: UPDATE/DELETE без WHERE

```sql
UPDATE students SET gpa = 0.0;  -- Змінює GPA ВСІХ студентів!
DELETE FROM students;            -- Видаляє ВСІХ студентів!
```

**Завжди** додавайте WHERE. **Завжди** тестуйте спочатку з SELECT.

## Розділ 3: Обмеження (Constraints)

Обмеження забезпечують якість даних:

- **NOT NULL** — стовпець обов'язковий
- **UNIQUE** — значення не повторюються
- **DEFAULT** — значення за замовчуванням
- **CHECK** — перевірка умови
- **FOREIGN KEY** — зв'язок з іншою таблицею

In [ ]:
# Видаляємо старі таблиці та створюємо нові з обмеженнями
conn.execute("DROP TABLE IF EXISTS students")
conn.commit()

createWithConstraints = """
CREATE TABLE students (
    studentId INTEGER PRIMARY KEY,
    firstName TEXT NOT NULL,
    lastName TEXT NOT NULL,
    email TEXT UNIQUE,
    grade INTEGER DEFAULT 9,
    gpa REAL CHECK(gpa >= 0.0 AND gpa <= 4.0)
);
"""

conn.execute(createWithConstraints)
conn.commit()
print("Таблицю 'students' створено з обмеженнями.")

In [ ]:
# Вставляємо коректні дані
conn.execute("INSERT INTO students (firstName, lastName, email, grade, gpa) VALUES ('Emma', 'Watson', 'emma@school.edu', 11, 3.85)")
conn.commit()

conn.execute("INSERT INTO students (firstName, lastName, email, grade, gpa) VALUES ('Frank', 'Miller', 'frank@school.edu', 10, 3.45)")
conn.commit()

# Без grade — буде DEFAULT 9
conn.execute("INSERT INTO students (firstName, lastName, email, gpa) VALUES ('Grace', 'Hopper', 'grace@school.edu', 3.95)")
conn.commit()

# Перевіряємо (зверніть увагу на grade Grace = 9)
result = pd.read_sql("SELECT * FROM students", conn)
print("Студенти з обмеженнями:")
print(result)

### Порушення обмежень

Спроби вставити некоректні дані будуть відхилені:

**NOT NULL:** `INSERT INTO students (email, gpa) VALUES ('no.name@school.edu', 3.50)` → Помилка: firstName обов'язковий

**UNIQUE:** `INSERT INTO students (firstName, lastName, email) VALUES ('Henry', 'Ford', 'emma@school.edu')` → Помилка: email вже існує

**CHECK:** `INSERT INTO students (firstName, lastName, email, gpa) VALUES ('Iris', 'Knight', 'iris@school.edu', 4.50)` → Помилка: GPA має бути 0.0-4.0

In [ ]:
# Створюємо таблицю з FOREIGN KEY
conn.execute("""
CREATE TABLE courses (
    courseId INTEGER PRIMARY KEY,
    courseName TEXT NOT NULL,
    credits INTEGER
)
""")
conn.commit()

conn.execute("""
CREATE TABLE enrollments (
    enrollmentId INTEGER PRIMARY KEY,
    studentId INTEGER NOT NULL,
    courseId INTEGER NOT NULL,
    FOREIGN KEY (studentId) REFERENCES students(studentId),
    FOREIGN KEY (courseId) REFERENCES courses(courseId)
)
""")
conn.commit()

print("Таблиці courses та enrollments створено з FOREIGN KEY.")

**Спробуй:** Вставити курси в таблицю courses, потім вставити записи в enrollments. Перевірити з SELECT.

In [ ]:
# Спробуй: Вставити курси та зарахування

## Підсумок

1. **CREATE TABLE та INSERT** — визначаємо структуру, вставляємо дані, зберігаємо з `conn.commit()`
2. **UPDATE та DELETE** — завжди тестуємо WHERE з SELECT спочатку!
3. **Обмеження** — NOT NULL, UNIQUE, DEFAULT, CHECK, FOREIGN KEY забезпечують якість даних

In [ ]:
conn.close()
print("З'єднання закрито.")